In [ ]:
# Get segment stats for building and WSF pixel counts

# ============================================================
# Calculate segment-level building and WSF variables
#
# Outputs:
#   1. integrated_building_count_2023
#      = count of building points per segment
#
#   2. wsf_pixel_count_2023
#      = count of 2023 WSF points per segment
#
#   3. building_wsf_ratio
#      = building point count / 2023 WSF point count
#
#   4. wsf_pixel_count_2026
#      = count of 2026 WSF points per segment
#
#   5. building_count_estimated_2026
#      = 2026 WSF count * building_wsf_ratio
# ============================================================

import arcpy
import os


# ============================================================
# INPUTS
# ============================================================

SEGMENT_GPKG = r"E:\WSF\wsf_work_arc\wsf_work_arc.gdb"

SEGMENTS = os.path.join(
    SEGMENT_GPKG,
    "segments_6_cities"
)


WORK_GDB2 = r"E:\WSF\wsf_work_arc\wsf_work_arc.gdb"


# ------------------------------------------------------------
# Building points
# ------------------------------------------------------------

BUILDING_POINTS = os.path.join(
    WORK_GDB2,
    "buildings_six_cities_points_2023_older"
)


# ------------------------------------------------------------
# WSF points - 2023
# ------------------------------------------------------------

WSF_POINTS_2023 = os.path.join(
    WORK_GDB2,
    "wsf_points_2023"
)


# ------------------------------------------------------------
# WSF points - 2026
# ------------------------------------------------------------

WSF_POINTS_2026 = os.path.join(
    WORK_GDB2,
    "wsf_pixel_count_2026"
)


# ============================================================
# OUTPUT FIELDS
# ============================================================

BUILDING_COUNT_FIELD = "integrated_building_count_2023"
WSF_COUNT_2023_FIELD = "wsf_pixel_count_2023"
RATIO_FIELD = "building_wsf_ratio"

WSF_COUNT_2026_FIELD = "wsf_pixel_count_2026"
ESTIMATED_BUILDING_2026_FIELD = "building_count_estimated_2026"


# ============================================================
# CHECK INPUTS
# ============================================================

print("Checking inputs...")

for path, name in [
    (SEGMENTS, "Segment layer"),
    (BUILDING_POINTS, "Building points"),
    (WSF_POINTS_2023, "2023 WSF points"),
    (WSF_POINTS_2026, "2026 WSF points")
]:

    if not arcpy.Exists(path):
        raise RuntimeError(
            f"{name} not found:\n{path}"
        )

print("All inputs found.")


# ============================================================
# SEGMENT ID FIELD
# ============================================================

SEGMENT_ID_FIELD = "OBJECTID"

segment_fields = [
    f.name
    for f in arcpy.ListFields(SEGMENTS)
]

if SEGMENT_ID_FIELD not in segment_fields:
    raise RuntimeError(
        f"Field '{SEGMENT_ID_FIELD}' was not found.\n\n"
        f"Available fields:\n"
        + "\n".join(segment_fields)
    )

print(
    f"Using segment ID field: {SEGMENT_ID_FIELD}"
)


# ============================================================
# ADD OUTPUT FIELDS
# ============================================================

existing_fields = [
    f.name
    for f in arcpy.ListFields(SEGMENTS)
]

for field_name, field_type in [

    (BUILDING_COUNT_FIELD, "LONG"),
    (WSF_COUNT_2023_FIELD, "LONG"),
    (RATIO_FIELD, "DOUBLE"),
    (WSF_COUNT_2026_FIELD, "LONG"),
    (ESTIMATED_BUILDING_2026_FIELD, "DOUBLE")

]:

    if field_name not in existing_fields:

        arcpy.management.AddField(
            SEGMENTS,
            field_name,
            field_type
        )

        print(
            f"Added field: {field_name}"
        )


# ============================================================
# TEMPORARY OUTPUTS
# ============================================================

BUILDING_COUNT_JOIN = os.path.join(
    WORK_GDB2,
    "temp_segment_building_counts"
)

WSF_COUNT_2023_JOIN = os.path.join(
    WORK_GDB2,
    "temp_segment_wsf_counts_2023"
)

WSF_COUNT_2026_JOIN = os.path.join(
    WORK_GDB2,
    "temp_segment_wsf_counts_2026"
)


for temp in [
    BUILDING_COUNT_JOIN,
    WSF_COUNT_2023_JOIN,
    WSF_COUNT_2026_JOIN
]:

    if arcpy.Exists(temp):

        print(
            f"Deleting existing temporary output: {temp}"
        )

        arcpy.management.Delete(temp)


# ============================================================
# 1. COUNT BUILDING POINTS
# ============================================================

print("\n============================================")
print("1. COUNTING BUILDING POINTS")
print("============================================")

arcpy.analysis.SpatialJoin(
    target_features=SEGMENTS,
    join_features=BUILDING_POINTS,
    out_feature_class=BUILDING_COUNT_JOIN,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="INTERSECT"
)


building_counts = {}

with arcpy.da.SearchCursor(
    BUILDING_COUNT_JOIN,
    [
        "TARGET_FID",
        "Join_Count"
    ]
) as cursor:

    for target_fid, count in cursor:

        building_counts[target_fid] = count


print(
    f"Counted buildings for "
    f"{len(building_counts):,} segments."
)


# ============================================================
# 2. COUNT 2023 WSF PIXELS
# ============================================================

print("\n============================================")
print("2. COUNTING 2023 WSF PIXELS")
print("============================================")

arcpy.analysis.SpatialJoin(
    target_features=SEGMENTS,
    join_features=WSF_POINTS_2023,
    out_feature_class=WSF_COUNT_2023_JOIN,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="INTERSECT"
)


wsf_counts_2023 = {}

with arcpy.da.SearchCursor(
    WSF_COUNT_2023_JOIN,
    [
        "TARGET_FID",
        "Join_Count"
    ]
) as cursor:

    for target_fid, count in cursor:

        wsf_counts_2023[target_fid] = count


print(
    f"Counted 2023 WSF pixels for "
    f"{len(wsf_counts_2023):,} segments."
)


# ============================================================
# 3. COUNT 2026 WSF PIXELS
# ============================================================

print("\n============================================")
print("3. COUNTING 2026 WSF PIXELS")
print("============================================")

arcpy.analysis.SpatialJoin(
    target_features=SEGMENTS,
    join_features=WSF_POINTS_2026,
    out_feature_class=WSF_COUNT_2026_JOIN,
    join_operation="JOIN_ONE_TO_ONE",
    join_type="KEEP_ALL",
    match_option="INTERSECT"
)


wsf_counts_2026 = {}

with arcpy.da.SearchCursor(
    WSF_COUNT_2026_JOIN,
    [
        "TARGET_FID",
        "Join_Count"
    ]
) as cursor:

    for target_fid, count in cursor:

        wsf_counts_2026[target_fid] = count


print(
    f"Counted 2026 WSF pixels for "
    f"{len(wsf_counts_2026):,} segments."
)


# ============================================================
# 4. UPDATE SEGMENTS
# ============================================================

print("\n============================================")
print("4. UPDATING SEGMENT ATTRIBUTES")
print("============================================")


with arcpy.da.UpdateCursor(
    SEGMENTS,
    [
        SEGMENT_ID_FIELD,
        BUILDING_COUNT_FIELD,
        WSF_COUNT_2023_FIELD,
        RATIO_FIELD,
        WSF_COUNT_2026_FIELD,
        ESTIMATED_BUILDING_2026_FIELD
    ]
) as cursor:

    for row in cursor:

        segment_id = row[0]


        # ----------------------------------------------------
        # 2023 building count
        # ----------------------------------------------------

        building_count = building_counts.get(
            segment_id,
            0
        )

        row[1] = building_count


        # ----------------------------------------------------
        # 2023 WSF count
        # ----------------------------------------------------

        wsf_count_2023 = wsf_counts_2023.get(
            segment_id,
            0
        )

        row[2] = wsf_count_2023


        # ----------------------------------------------------
        # Building / WSF ratio
        # ----------------------------------------------------

        if wsf_count_2023 > 0:

            building_wsf_ratio = (
                building_count /
                wsf_count_2023
            )

            row[3] = building_wsf_ratio

        else:

            building_wsf_ratio = None
            row[3] = None


        # ----------------------------------------------------
        # 2026 WSF count
        # ----------------------------------------------------

        wsf_count_2026 = wsf_counts_2026.get(
            segment_id,
            0
        )

        row[4] = wsf_count_2026


        # ----------------------------------------------------
        # Estimated 2026 building count
        #
        # 2026 WSF count * 2023 building/WSF ratio
        # ----------------------------------------------------

        if (
            wsf_count_2026 > 0
            and building_wsf_ratio is not None
        ):

            row[5] = (
                wsf_count_2026 *
                building_wsf_ratio
            )

        else:

            row[5] = None


        cursor.updateRow(row)


# ============================================================
# 5. CLEAN UP
# ============================================================

print("\nCleaning up temporary outputs...")

for temp in [
    BUILDING_COUNT_JOIN,
    WSF_COUNT_2023_JOIN,
    WSF_COUNT_2026_JOIN
]:

    if arcpy.Exists(temp):

        arcpy.management.Delete(temp)


# ============================================================
# COMPLETE
# ============================================================

print("\n============================================")
print("DONE")
print("============================================")

print("\nUpdated segment layer:")
print(SEGMENTS)

print("\nCreated/updated attributes:")

print(
    f" - {BUILDING_COUNT_FIELD}"
)

print(
    f" - {WSF_COUNT_2023_FIELD}"
)

print(
    f" - {RATIO_FIELD}"
)

print(
    f" - {WSF_COUNT_2026_FIELD}"
)

print(
    f" - {ESTIMATED_BUILDING_2026_FIELD}"
)

print("\nEstimated 2026 building count:")
print(
    "  wsf_pixel_count_2026 * building_wsf_ratio"
)

Checking inputs...
All inputs found.
Using segment ID field: OBJECTID
Added field: wsf_pixel_count_2026
Added field: building_count_estimated_2026
Deleting existing temporary output: E:\WSF\wsf_work_arc\wsf_work_arc.gdb\temp_segment_building_counts

1. COUNTING BUILDING POINTS
